# Workshop 1 — Your First Analysis Script
### Does reward change how people perform? A look at a (fake) reward task

**How this notebook works**
- A notebook is made of **cells**. Grey boxes are code; everything else is notes.
- Click a code cell and press **Shift + Enter** to run it. The output appears underneath.
- Run cells **in order, top to bottom**. Later cells depend on earlier ones.
- Cells marked `# TODO` have a blank `___` for you to fill in.
- Errors are normal! Read the **last line** of the red message — it usually tells you what went wrong.

**First thing:** click **File → Save a copy in Drive** so you have your own editable copy.


---
## Part 0 — Python in 5 minutes

Python reads your code **one line at a time, top to bottom**. Let's learn four ideas: *variables*, *lists*, *loops*, and *if-statements*.

In [ ]:
# Anything after a # is a COMMENT. Python ignores it — it's a note for humans.

# A VARIABLE is a name that stores a value
n_participants = 20
task_name = "reward task"

print(task_name, "has", n_participants, "participants")

In [ ]:
# A LIST holds several values, in order
rts = [512, 634, 598, 2950, 101]   # five reaction times, in milliseconds

print(len(rts))    # how many items?
print(max(rts))    # biggest value
print(rts[0])      # the FIRST item — Python starts counting at 0!

In [ ]:
# TODO: print the smallest value in rts (hint: the opposite of max)
print(___)

> 🤔 **Think:** Two of those reaction times look suspicious. Which ones, and why? Could a human really respond in 101 ms? Keep that thought — we'll come back to it.

In [ ]:
# A FOR-LOOP repeats code for every item in a list.
# An IF-STATEMENT only runs code when a condition is True.
# The INDENTATION (4 spaces) tells Python which lines belong inside the loop / if.

for rt in rts:
    if rt < 150:
        print(rt, "-> too fast to be a real response")
    else:
        print(rt, "-> looks OK")

In [ ]:
# TODO: copy the loop above and add an `elif` line so that anything
#       slower than 2500 ms prints "-> suspiciously slow"
for rt in rts:
    if rt < 150:
        print(rt, "-> too fast to be a real response")
    elif ___:
        print(rt, "-> suspiciously slow")
    else:
        print(rt, "-> looks OK")

---
## Part 1 — Bring the data in

Real data has thousands of rows, so we use **libraries** — toolboxes other people wrote.
- **pandas** handles tables (think: Excel inside Python)
- **matplotlib** makes plots

In [ ]:
import pandas as pd              # "pd" is a nickname so we type less
import matplotlib.pyplot as plt  # "plt" is the plotting nickname

In [ ]:
# The data file lives online, so nobody needs to download anything.
DATA_URL = "https://raw.githubusercontent.com/slopez4/RA_Workshops/main/01_intro_python_analysis/data/reward_task_mock.csv"

df = pd.read_csv(DATA_URL)   # df = "data frame" = a table
df.head()                    # show the first 5 rows

### Codebook — what each column means

| column | meaning |
|---|---|
| `participant_id` | who (P01–P20) |
| `order` | did they get the reward block first or second? (counterbalancing) |
| `block` | 1–4. Each block is 20 trials |
| `trial` | trial number within the block (1–20) |
| `condition` | `reward` = earned points for fast, correct answers; `no_reward` = no points |
| `rt_ms` | reaction time in milliseconds (empty = no response) |
| `correct` | 1 = correct, 0 = wrong or missed |
| `motivation_rating` | "How motivated were you?" (1–7), asked **once at the end of each block** |

> 🤔 **Think:** What does **one row** of this table represent? A participant? A block? A trial?

In [ ]:
# TODO: how many rows and columns are there? Use df.shape
df.___

In [ ]:
# How many different participants? And how many rows does each one have?
print(df["participant_id"].nunique(), "participants")
df["participant_id"].value_counts()

> 🤔 **Think:** 20 participants × 4 blocks × 20 trials = ? Does that match `df.shape`? Checking that the numbers add up is the **first** thing to do with any new dataset.

---
## Part 2 — Look before you analyze

Before asking "did reward work?", ask **"can I trust this data?"**

In [ ]:
# Summary statistics for every numeric column
df.describe()

> 🤔 **Think:** Look at the `rt_ms` row.
> - What's the **min**? The **max**? Are those believable?
> - The `count` for `rt_ms` is smaller than for the other columns. Why?

In [ ]:
# Count missing values (NaN = "Not a Number" = empty cell) in each column
df.isna().sum()

In [ ]:
# A histogram shows how the reaction times are spread out
df["rt_ms"].hist(bins=60)
plt.xlabel("Reaction time (ms)")
plt.ylabel("Number of trials")
plt.title("All reaction times")
plt.show()

In [ ]:
# GROUPBY = split the table by participant, then compute something for each piece.
# The mean of a 0/1 column is the proportion of 1s -> accuracy!
acc_by_participant = df.groupby("participant_id")["correct"].mean()
acc_by_participant.sort_values()

> 🤔 **Think:** Does anyone stand out? The task has two response buttons, so what accuracy would you get by **guessing**? What should we do with this person — and **why** is that decision better made *before* looking at the results?

---
## Part 3 — Clean the data

A **condition** like `df["rt_ms"] < 150` gives `True`/`False` for every row. We can count the `True`s or use them to keep only certain rows.

In [ ]:
too_fast = df["rt_ms"] < 150
print(too_fast.sum(), "trials faster than 150 ms")   # True counts as 1, False as 0

In [ ]:
# TODO: count the trials slower than 2500 ms
too_slow = ___
print(too_slow.sum(), "trials slower than 2500 ms")

In [ ]:
# Write the RULE, not the name: anyone below 60% accuracy is excluded.
# If new participants are added later, the rule still works.
excluded = acc_by_participant[acc_by_participant < 0.60].index.tolist()
print("Excluding:", excluded)

# Keep only the rows we trust.   &  means "and"    ~  means "not"
clean = df[
    ~df["participant_id"].isin(excluded)
    & (df["rt_ms"] >= 150)
    & (df["rt_ms"] <= 2500)
].copy()

print(len(df), "rows before cleaning")
print(len(clean), "rows after cleaning")

> 🤔 **Think:** We never wrote a line to remove the **missing** RTs, but they're gone. Why? (Hint: is an empty cell `>= 150`?) Is dropping missed responses the right call when we compute **accuracy**?

---
## Part 4 — Does reward change behavior?

In [ ]:
# Average RT and accuracy for each condition
clean.groupby("condition")[["rt_ms", "correct"]].mean()

That averages over *trials*. Better: first get **one number per person per condition**, so every participant counts equally. `unstack()` turns the condition labels into columns (like a pivot table in Excel).

In [ ]:
per_person = clean.groupby(["participant_id", "condition"])["rt_ms"].mean().unstack()

# A new column: reward RT minus no-reward RT. Negative = faster with reward.
per_person["reward_effect"] = per_person["reward"] - per_person["no_reward"]
per_person.round(0)

In [ ]:
# TODO: how many people were FASTER with reward? (hint: reward_effect < 0)
n_faster = (___).sum()
print(n_faster, "of", len(per_person), "participants were faster with reward")

In [ ]:
# One line per participant: a for-loop doing real work!
for pid, row in per_person.iterrows():
    plt.plot(["no_reward", "reward"], [row["no_reward"], row["reward"]],
             color="gray", alpha=0.5, marker="o")

# The group average on top, in a thick line
plt.plot(["no_reward", "reward"],
         [per_person["no_reward"].mean(), per_person["reward"].mean()],
         color="black", linewidth=3, marker="o")

plt.ylabel("Mean reaction time (ms)")
plt.title("Each gray line is one participant")
plt.show()

> 🤔 **Think:** Does *everyone* show the effect? What does the gray-line plot tell you that a single average can't?

---
## Part 5 — Does the effect change over time? *(bridge to today's discussion)*

Each half of the session (blocks 1–2 and blocks 3–4) contains one reward and one no-reward block for every participant. So we can ask: **is reward as powerful at the end as at the start?**

A **dictionary** (`{key: value}`) is a lookup table — here it maps each block to its half.

In [ ]:
half_of_block = {1: "1st half", 2: "1st half", 3: "2nd half", 4: "2nd half"}
clean["half"] = clean["block"].map(half_of_block)

by_half = clean.groupby(["participant_id", "half", "condition"])["rt_ms"].mean().unstack()
by_half["reward_effect"] = by_half["reward"] - by_half["no_reward"]

by_half.groupby("half")[["no_reward", "reward", "reward_effect"]].mean().round(0)

In [ ]:
# Motivation was rated ONCE per block but copied onto all 20 rows of that block.
# drop_duplicates keeps one row per participant per block, so we don't over-count.
ratings = clean.drop_duplicates(["participant_id", "block"])

motivation = ratings.groupby(["half", "condition"])["motivation_rating"].mean().unstack()
motivation.plot(kind="bar", rot=0)
plt.ylabel("Mean motivation rating (1-7)")
plt.title("Self-reported motivation")
plt.show()
motivation.round(2)

> 🤔 **Discussion — motivational shifts:**
> 1. The reward advantage roughly **halves** in the second half. List as many explanations as you can. (Fatigue? Getting used to the reward? Practice making everyone fast anyway? Something about the *no-reward* blocks?)
> 2. Look at the motivation ratings. Which condition changed more — reward or no-reward? Does that change your explanation?
> 3. What would you add to the design to tell these explanations apart?

In [ ]:
# ⭐ STRETCH (if you finish early): repeat Part 5 for ACCURACY instead of RT.
#    Copy the by_half cell and change "rt_ms" to "correct".


---
## Wrap-up — from notebook to script

Everything you ran today, in order, **is** an analysis script. The file [`analysis.py`](https://github.com/slopez4/RA_Workshops/blob/main/01_intro_python_analysis/analysis.py) is the same code with the teaching bits removed: load → check → clean → analyze → plot.

**The habits that matter more than the syntax:**
1. Check the numbers add up (rows, participants, trials).
2. Look at distributions before computing averages.
3. Decide exclusion rules *before* looking at results, and write them as rules.
4. Summarize per participant first, then across participants.
5. Plot individuals, not just means.